In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import TimestampType, IntegerType, DateType
from pyspark.sql.functions import col, initcap, trim, when


def normalize_grid_events(df: DataFrame) -> DataFrame:
    return ( df
            .withColumn("last_updated_ts", col('last_updated_ts').cast(TimestampType()))
            .withColumn("event_date", col('event_ts').cast(DateType()))
            .withColumn("region", trim(col('region')))
            .withColumn("event_type", trim(col('event_type')))
            .withColumn("severity", trim(col('severity')))
            .withColumn("duration_minutes", col('duration_minutes').cast(IntegerType()))
            .drop(col('_rescued_data'), col('event_ts'))
    )


def derived_critical_duration(df: DataFrame) -> DataFrame:
    quantiles = df.approxQuantile("duration_minutes", [0.25, 0.5, 0.75], 0.0)
    q1, q2, q3 = quantiles[0], quantiles[1], quantiles[2]

    return ( df.withColumn('waiting_time',
        when(col('duration_minutes') > q3, "Long")
        .when(col('duration_minutes') > q2, "Average")
        .when(col('duration_minutes') > q1, "Short")
        .otherwise("Extreme Short")
    ))